In [1]:
from pyspark.sql import Window, functions as F
from delta.tables import DeltaTable

config = {
  "agent":    {"key":"AgentID","wm":"LastActiveDate","fmt":"M/d/yyyy","dates":["JoinDate","LastActiveDate"]},
  "builder":  {"key":"BuilderID","wm":None,"fmt":None,"dates":[]},
  "customer": {"key":"CustomerID","wm":"LastInteractionDate","fmt":"yyyy-MM-dd","dates":["FirstContactDate","LastInteractionDate"]},
  "expense":  {"key":"ExpenseID","wm":"ExpenseDate","fmt":"yyyy-MM-dd","dates":["ExpenseDate"]},
  "leads":    {"key":"LeadID","wm":"LastFollowUpDate","fmt":"yyyy-MM-dd","dates":["LeadCreatedDate","FirstContactDate","LastFollowUpDate","ConversionDate"]},
  "lease":    {"key":"LeaseID","wm":"AgreementDate","fmt":"yyyy-MM-dd","dates":["LeaseStartDate","LeaseEndDate","AgreementDate"]},
  "property": {"key":"PropertyID","wm":"ListingDate","fmt":"yyyy-MM-dd","dates":["LaunchDate","PossessionDate","ListingDate"]},
  "sales":    {"key":"SalesID","wm":"SaleDate","fmt":"yyyy-MM-dd","dates":["SaleDate","AgreementDate","RegistrationDate"]},
}

for t, c in config.items():
    if not spark.catalog.tableExists(f"bronze_{t}"): continue
    df = spark.table(f"bronze_{t}")
    for d in c["dates"]:                                       # type conversions
        if d in df.columns: df = df.withColumn(d, F.to_date(F.col(d).cast("string"), c["fmt"]))
    for col in df.columns:                                     # null handling: blank -> null
        df = df.withColumn(col, F.when(F.trim(F.col(col).cast("string"))=="", None).otherwise(F.col(col)))
    order = F.col(c["wm"]).desc_nulls_last() if c["wm"] else F.col("_ingested_at").desc()
    w = Window.partitionBy(c["key"]).orderBy(order, F.col("_ingested_at").desc())
    df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")   # dedup: latest per key

    tgt = f"silver_{t}"
    if not spark.catalog.tableExists(tgt):
        df.write.format("delta").saveAsTable(tgt); print("init", tgt); continue
    dt = DeltaTable.forName(spark, tgt)
    m = dt.alias("t").merge(df.alias("s"), f"t.{c['key']} = s.{c['key']}")
    m = (m.whenMatchedUpdateAll(condition=f"s.{c['wm']} > t.{c['wm']}") if c["wm"] else m.whenMatchedUpdateAll())
    m.whenNotMatchedInsertAll().execute()
    print("merged", tgt)


StatementMeta(, 004c5e08-5e89-4f7d-9e43-096eaa7b323c, 3, Finished, Available, Finished, False)

init silver_agent
init silver_builder
init silver_customer
init silver_expense
init silver_leads
init silver_lease
init silver_property
init silver_sales


In [2]:
spark.sql("SELECT AgentID, JoinDate, LastActiveDate FROM silver_agent LIMIT 5").show()

StatementMeta(, 004c5e08-5e89-4f7d-9e43-096eaa7b323c, 4, Finished, Available, Finished, False)

+-------+----------+--------------+
|AgentID|  JoinDate|LastActiveDate|
+-------+----------+--------------+
|      1|2020-01-05|    2025-01-01|
|      2|2020-01-18|    2025-01-02|
|      3|2020-02-02|    2025-01-03|
|      4|2020-02-15|    2025-01-04|
|      5|2020-02-28|    2025-01-05|
+-------+----------+--------------+

